# Pengayaan 3 — Membangun Model Machine Learning

Di sesi pengayaan ini kita akan menggunakan dataset hasil preprocessing dari Day 2 untuk **membangun dan mengevaluasi model machine learning** yang memprediksi **churn pelanggan telekomunikasi**.

> **Churn** = pelanggan yang berhenti berlangganan. Prediksi churn memungkinkan perusahaan untuk proaktif memberikan retensi kepada pelanggan berisiko tinggi sebelum mereka pergi.

---

## Agenda

1. **Load data** hasil preprocessing
2. **Train-test split** — memisahkan data latih dan uji
3. **Membangun model** — tiga algoritma populer
   - Logistic Regression
   - Decision Tree
   - Random Forest
4. **Evaluasi model** — Accuracy, Precision, Recall, F1, ROC-AUC
5. **Perbandingan model** — mana yang terbaik dan mengapa?
6. **Interpretasi** — fitur apa yang paling berpengaruh?
7. **Prediksi data baru** — simulasi penggunaan model di produksi

---

**Prasyarat:** Sudah menyelesaikan `day-1.ipynb` dan `day-2.ipynb`. File `data/data_preprocessing_result.parquet` harus sudah ada.

## 0. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data hasil preprocessing
df = pd.read_parquet('data/data_preprocessing_result.parquet')

print('Shape:', df.shape)
print('Kolom:', df.columns.tolist())
print()
print('Distribusi target (Churn):')
print(df['Churn'].value_counts())
print(f'Churn rate: {df["Churn"].mean():.1%}')

In [ ]:
df.head()

---

## 1. Persiapan Data — Train-Test Split

Sebelum melatih model, kita harus memisahkan data menjadi dua bagian:

- **Training set** — data yang digunakan untuk melatih model (model "belajar" dari data ini)
- **Test set** — data yang *tidak pernah dilihat* model selama training, digunakan untuk evaluasi yang fair

Rasio yang umum digunakan: **80% training, 20% test**.

> **Mengapa tidak langsung pakai semua data untuk training?**  
> Jika model dievaluasi pada data yang sama dengan data latihnya, hasilnya akan selalu bagus — tapi model mungkin hanya *menghafal* data, bukan *belajar pola*. Test set mengukur kemampuan generalisasi model ke data yang belum pernah dilihat.

In [ ]:
from sklearn.model_selection import train_test_split

# Pisahkan fitur (X) dan target (y)
# Gunakan fitur yang sudah di-scale untuk konsistensi
feature_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'PaperlessBilling', 'Contract_encoded', 'is_weekend_signup',
    'signup_month', 'signup_quarter',
    'tenure_scaled', 'MonthlyCharges_scaled', 'TotalCharges_scaled',
    'AvgMonthlyCharge_scaled', 'ChargeDeviation_scaled', 'days_active_scaled'
]

X = df[feature_cols]
y = df['Churn']

# Split: 80% train, 20% test
# stratify=y memastikan proporsi churn di train dan test sama
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set  : {X_train.shape[0]:,} baris ({X_train.shape[0]/len(X):.0%})')
print(f'Test set      : {X_test.shape[0]:,} baris ({X_test.shape[0]/len(X):.0%})')
print()
print('Proporsi Churn di training set:')
print(y_train.value_counts(normalize=True).round(3))
print('Proporsi Churn di test set:')
print(y_test.value_counts(normalize=True).round(3))

---

## 2. Membangun Model

Kita akan mencoba tiga algoritma klasifikasi yang populer dan memiliki karakteristik berbeda:

| Algoritma | Cara Kerja | Kelebihan | Kekurangan |
|---|---|---|---|
| **Logistic Regression** | Garis pemisah linier dengan fungsi sigmoid | Cepat, mudah diinterpretasi, baseline yang baik | Kurang baik jika hubungan non-linier |
| **Decision Tree** | Membuat aturan if-then berdasarkan fitur | Sangat mudah diinterpretasi, bisa capture non-linier | Mudah overfit |
| **Random Forest** | Ensemble banyak decision tree | Akurat, robust terhadap overfit | Lebih lambat, kurang interpretable |

### 2a. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

# Latih model
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Prediksi
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]  # probabilitas kelas 1 (Churned)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['Stayed (0)', 'Churned (1)']))

### 2b. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# max_depth=5 mencegah pohon terlalu dalam (overfit)
dt = DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print('=== Decision Tree ===')
print(classification_report(y_test, y_pred_dt, target_names=['Stayed (0)', 'Churned (1)']))

### 2c. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Stayed (0)', 'Churned (1)']))

---

## 3. Evaluasi Model

### Memahami Metrik Evaluasi

Untuk kasus churn prediction, **tidak semua error sama berbahayanya**:

| | Prediksi: Stayed | Prediksi: Churned |
|---|---|---|
| **Aktual: Stayed** | True Negative (TN) ✓ | False Positive (FP) — biaya retensi sia-sia |
| **Aktual: Churned** | False Negative (FN) ❌ — pelanggan hilang! | True Positive (TP) ✓ |

- **False Negative (FN)** jauh lebih berbahaya: kita *gagal mendeteksi* pelanggan yang sebenarnya mau churn → kehilangan revenue
- **Recall** (sensitivity) mengukur seberapa baik kita menangkap pelanggan yang benar-benar churn: `TP / (TP + FN)`
- **Precision** mengukur akurasi prediksi churn kita: `TP / (TP + FP)`
- **F1-Score** adalah harmonic mean Precision dan Recall — seimbang antara keduanya
- **ROC-AUC** mengukur kemampuan model membedakan dua kelas secara keseluruhan

In [ ]:
# Hitung semua metrik untuk ketiga model
models = {
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Decision Tree':       (y_pred_dt, y_prob_dt),
    'Random Forest':       (y_pred_rf, y_prob_rf),
}

results = []
for name, (y_pred, y_prob) in models.items():
    results.append({
        'Model': name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1-Score':  f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index('Model')
print('Perbandingan Metrik Evaluasi:')
print(results_df.round(4).to_string())

In [ ]:
# Visualisasi perbandingan metrik
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart metrik
results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']].plot(
    kind='bar', ax=axes[0], colormap='Set2', width=0.7
)
axes[0].set_title('Perbandingan Metrik Evaluasi')
axes[0].set_xlabel('')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(loc='lower right')
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.2f', fontsize=7, padding=2)

# ROC Curve
from sklearn.metrics import roc_curve
colors = ['#3498db', '#e67e22', '#27ae60']
for (name, (_, y_prob)), color in zip(models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix untuk ketiga model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (y_pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Stayed', 'Churned'],
        yticklabels=['Stayed', 'Churned']
    )
    ax.set_title(f'{name}\nConfusion Matrix')
    ax.set_xlabel('Prediksi')
    ax.set_ylabel('Aktual')

    # Tambahkan keterangan FN (kiri bawah = berbahaya)
    fn = cm[1, 0]
    ax.text(0.5, 1.35, f'False Negative = {fn}', ha='center',
            transform=ax.transAxes, color='red', fontsize=9)

plt.tight_layout()
plt.show()

### Interpretasi Hasil

Perhatikan **Confusion Matrix** dan **False Negative** masing-masing model:

- **False Negative** = pelanggan yang sebenarnya churn tapi kita prediksi *tidak churn* → kita tidak melakukan intervensi → pelanggan hilang
- Dalam kasus bisnis ini, **Recall** (kemampuan mendeteksi churn) lebih penting dari Precision

> Model dengan **Recall tertinggi** adalah yang paling baik untuk kasus ini, selama Precision-nya masih masuk akal.

---

## 4. Interpretasi — Fitur yang Paling Berpengaruh

Memahami fitur apa yang paling berpengaruh membantu tim bisnis untuk:
- Mengidentifikasi **faktor risiko churn**
- Merancang **strategi retensi** yang tepat sasaran

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Random Forest Feature Importance ---
rf_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
rf_imp.head(10).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].invert_yaxis()
axes[0].set_title('Random Forest\nFeature Importance (Top 10)')
axes[0].set_xlabel('Importance Score')

# --- Logistic Regression Coefficients ---
lr_coef = pd.Series(lr.coef_[0], index=feature_cols).sort_values(key=abs, ascending=False)
top10_lr = lr_coef.head(10)
colors_lr = ['#e74c3c' if v > 0 else '#27ae60' for v in top10_lr.values]
top10_lr.plot(kind='barh', ax=axes[1], color=colors_lr)
axes[1].invert_yaxis()
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Logistic Regression\nKoefisien (Top 10 |koefisien| terbesar)\nMerah = meningkatkan churn, Hijau = mengurangi churn')
axes[1].set_xlabel('Koefisien')

plt.tight_layout()
plt.show()

In [ ]:
# Insight dari koefisien Logistic Regression
print('=== Faktor yang MENINGKATKAN risiko churn (koefisien positif) ===')
positive_coef = lr_coef[lr_coef > 0].sort_values(ascending=False).head(8)
for feat, coef in positive_coef.items():
    print(f'  {feat:<30} : +{coef:.4f}')

print()
print('=== Faktor yang MENGURANGI risiko churn (koefisien negatif) ===')
negative_coef = lr_coef[lr_coef < 0].sort_values().head(8)
for feat, coef in negative_coef.items():
    print(f'  {feat:<30} : {coef:.4f}')

---

## 5. Visualisasi Decision Tree

Salah satu keunggulan Decision Tree adalah kemudahannya untuk divisualisasikan dan dijelaskan kepada stakeholder non-teknis.

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(20, 8))
plot_tree(
    dt,
    feature_names=feature_cols,
    class_names=['Stayed', 'Churned'],
    filled=True,
    max_depth=3,   # tampilkan 3 level teratas agar tidak terlalu padat
    fontsize=9,
    rounded=True,
    impurity=False
)
plt.title('Decision Tree — 3 Level Teratas\n(Biru = Stayed, Oranye = Churned)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Cetak aturan keputusan dari Decision Tree dalam bentuk teks
from sklearn.tree import export_text

tree_rules = export_text(dt, feature_names=feature_cols, max_depth=3)
print('Aturan Keputusan Decision Tree (3 Level Pertama):')
print(tree_rules[:3000])  # tampilkan 3000 karakter pertama

---

## 6. Prediksi Data Baru

Simulasi penggunaan model di production: kita punya data pelanggan baru yang **belum pernah dilihat model**, dan kita ingin memprediksi kemungkinan churn mereka.

In [ ]:
# Buat 5 profil pelanggan baru (data sintetis)
new_customers = pd.DataFrame([
    # Pelanggan 1: baru bergabung, tagihan tinggi, month-to-month
    {'gender': 1, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0,
     'PhoneService': 1, 'PaperlessBilling': 1, 'Contract_encoded': 0,
     'is_weekend_signup': 0, 'signup_month': 3, 'signup_quarter': 1,
     'tenure_scaled': -1.2, 'MonthlyCharges_scaled': 1.5,
     'TotalCharges_scaled': -0.9, 'AvgMonthlyCharge_scaled': 1.5,
     'ChargeDeviation_scaled': 0.1, 'days_active_scaled': -1.2},

    # Pelanggan 2: sudah 5 tahun, kontrak 2 tahun, tagihan menengah
    {'gender': 0, 'SeniorCitizen': 0, 'Partner': 1, 'Dependents': 1,
     'PhoneService': 1, 'PaperlessBilling': 0, 'Contract_encoded': 2,
     'is_weekend_signup': 0, 'signup_month': 6, 'signup_quarter': 2,
     'tenure_scaled': 1.5, 'MonthlyCharges_scaled': -0.2,
     'TotalCharges_scaled': 2.0, 'AvgMonthlyCharge_scaled': -0.1,
     'ChargeDeviation_scaled': -0.1, 'days_active_scaled': 1.5},

    # Pelanggan 3: senior, baru 3 bulan, tidak punya partner
    {'gender': 1, 'SeniorCitizen': 1, 'Partner': 0, 'Dependents': 0,
     'PhoneService': 1, 'PaperlessBilling': 1, 'Contract_encoded': 0,
     'is_weekend_signup': 1, 'signup_month': 11, 'signup_quarter': 4,
     'tenure_scaled': -1.4, 'MonthlyCharges_scaled': 1.8,
     'TotalCharges_scaled': -1.0, 'AvgMonthlyCharge_scaled': 1.8,
     'ChargeDeviation_scaled': 0.3, 'days_active_scaled': -1.4},

    # Pelanggan 4: loyal 4 tahun, kontrak 1 tahun, tagihan rendah
    {'gender': 0, 'SeniorCitizen': 0, 'Partner': 1, 'Dependents': 0,
     'PhoneService': 0, 'PaperlessBilling': 0, 'Contract_encoded': 1,
     'is_weekend_signup': 0, 'signup_month': 1, 'signup_quarter': 1,
     'tenure_scaled': 1.2, 'MonthlyCharges_scaled': -1.1,
     'TotalCharges_scaled': 1.1, 'AvgMonthlyCharge_scaled': -1.0,
     'ChargeDeviation_scaled': -0.1, 'days_active_scaled': 1.2},

    # Pelanggan 5: 1 tahun, month-to-month, tagihan naik drastis
    {'gender': 1, 'SeniorCitizen': 0, 'Partner': 0, 'Dependents': 0,
     'PhoneService': 1, 'PaperlessBilling': 1, 'Contract_encoded': 0,
     'is_weekend_signup': 0, 'signup_month': 8, 'signup_quarter': 3,
     'tenure_scaled': -0.5, 'MonthlyCharges_scaled': 2.1,
     'TotalCharges_scaled': -0.3, 'AvgMonthlyCharge_scaled': -0.2,
     'ChargeDeviation_scaled': 2.3, 'days_active_scaled': -0.5},
])

# Deskripsi profil untuk referensi
profiles = [
    'Baru bergabung, tagihan tinggi, month-to-month',
    'Pelanggan loyal 5 tahun, kontrak 2 tahun',
    'Senior, baru 3 bulan, tagihan tinggi',
    'Loyal 4 tahun, kontrak 1 tahun, tagihan rendah',
    'Setahun, tagihan naik drastis bulan ini',
]

print('Data pelanggan baru (fitur yang relevan):')
new_customers[feature_cols].head()

In [ ]:
# Prediksi menggunakan Random Forest (model terbaik)
pred_labels = rf.predict(new_customers[feature_cols])
pred_proba  = rf.predict_proba(new_customers[feature_cols])[:, 1]

# Buat tabel hasil prediksi
pred_result = pd.DataFrame({
    'Pelanggan': [f'Pelanggan {i+1}' for i in range(len(new_customers))],
    'Profil': profiles,
    'Prediksi': ['Churned ⚠️' if p == 1 else 'Stayed ✓' for p in pred_labels],
    'Prob. Churn (%)': (pred_proba * 100).round(1),
    'Risiko': pd.cut(pred_proba, bins=[0, 0.3, 0.6, 1.0], labels=['Rendah', 'Menengah', 'Tinggi'])
})

print('Hasil Prediksi Churn:')
print(pred_result.to_string(index=False))

In [ ]:
# Visualisasi probabilitas churn per pelanggan
fig, ax = plt.subplots(figsize=(10, 4))

colors = ['#e74c3c' if p > 0.6 else '#f39c12' if p > 0.3 else '#27ae60' for p in pred_proba]
bars = ax.barh(
    [f'Pelanggan {i+1}\n({p[:40]})' for i, p in enumerate(profiles)],
    pred_proba * 100,
    color=colors
)

ax.axvline(x=30, color='orange', linestyle='--', alpha=0.7, label='Threshold Menengah (30%)')
ax.axvline(x=60, color='red',    linestyle='--', alpha=0.7, label='Threshold Tinggi (60%)')
ax.set_xlabel('Probabilitas Churn (%)')
ax.set_title('Prediksi Risiko Churn Pelanggan Baru\n(Random Forest)')
ax.set_xlim(0, 100)
ax.legend()

for bar, prob in zip(bars, pred_proba):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{prob*100:.1f}%', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 7. Ringkasan & Rekomendasi Bisnis

### Hasil Perbandingan Model

In [ ]:
# Tampilkan tabel perbandingan akhir
print('=== Perbandingan Final Model ===')
print(results_df.round(4).to_string())
print()

best_recall = results_df['Recall'].idxmax()
best_auc    = results_df['ROC-AUC'].idxmax()
best_f1     = results_df['F1-Score'].idxmax()

print(f'Model dengan Recall tertinggi  : {best_recall} ({results_df.loc[best_recall, "Recall"]:.4f})')
print(f'Model dengan ROC-AUC tertinggi : {best_auc} ({results_df.loc[best_auc, "ROC-AUC"]:.4f})')
print(f'Model dengan F1-Score tertinggi: {best_f1} ({results_df.loc[best_f1, "F1-Score"]:.4f}')
print()
print('Rekomendasi: Untuk kasus churn prediction, prioritaskan model dengan Recall tertinggi')
print(f'→ Gunakan {best_recall} sebagai model utama')

### Alur Lengkap yang Telah Kita Pelajari

```
Data Mentah (telco_raw.csv)
    ↓
[Day 1] EDA → pahami data
         Data Cleansing → tangani missing values, outlier, imputation
    ↓
[Day 2] Feature Extraction → gali informasi baru (teks, numerik, datetime)
         Feature Transformation → scaling, binning, encoding
         Feature Selection → pilih fitur paling relevan
         Export → data/data_preprocessing_result.parquet
    ↓
[Pengayaan 3] Train-Test Split
               Build Model (LR / DT / RF)
               Evaluate (Accuracy, Precision, Recall, F1, AUC)
               Interpret (Feature Importance, Decision Rules)
               Predict (data baru)
```

### Key Takeaways

| Poin | Penjelasan |
|---|---|
| **Pilih metrik sesuai konteks bisnis** | Untuk churn, Recall lebih penting dari Accuracy |
| **Tidak ada model yang selalu terbaik** | Setiap algoritma memiliki trade-off |
| **Interpretabilitas penting** | Logistic Regression dan Decision Tree lebih mudah dijelaskan ke stakeholder |
| **Feature engineering menentukan kualitas model** | Garbage in, garbage out — model sebaik fitur yang diberikan |
| **Selalu validasi dengan data yang belum pernah dilihat** | Test set dan data produksi baru adalah ujian sesungguhnya |